In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path("/home/mersad/protBuild")

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

In [2]:
import os
import json
import torch
import multiprocessing as mp
from torch.utils.data import DataLoader
from ml.src.training.dataset.protein_dataset import ProteinDataset
from ml.src.models.tokenizer.bpe import BPETokenizer
from ml.src.models.gpt2.gpt2 import GPT2
from ml.src.models.llama2.llama2 import LLAMA2
from ml.src.training.trainer.train import train_model
from torch.optim.lr_scheduler import CosineAnnealingLR


In [3]:
config = {}
# config_path = PROJECT_ROOT / 'ml/src/models/configs/gpt2-config.json'
config_path = PROJECT_ROOT / 'ml/src/models/configs/llama2-config.json'

selected_config = 'small'
with open (config_path, 'r') as f:
    data = json.load(f)
    if selected_config in data:
        config = data[selected_config]
    else:
        raise KeyError(f"Configuration '{selected_config}' not found.")


In [4]:
vocab_path = PROJECT_ROOT /'ml/datasets/converted/sequences/tokenizer/uniprot_bpe_32000_vocab.json'
merges_path = PROJECT_ROOT /'ml/datasets/converted/sequences/tokenizer/uniprot_bpe_32000_merges.json'
train_corpus_path = PROJECT_ROOT /'ml/datasets/converted/sequences/uniprot_sprot_train.txt'
val_corpus_path = PROJECT_ROOT /'ml/datasets/converted/sequences/uniprot_sprot_val.txt'
end_token='<|endofprotein|>'
checkpoint_path=PROJECT_ROOT /'ml/notebooks/checkpoints/'
save_file_name='checkpoint-model.pth'
device = 'cuda' if torch.cuda.is_available() else 'cpu'
tokenizer = BPETokenizer().load_vocab_and_merges(vocab_path, merges_path)


In [5]:
torch.manual_seed(123)

In [6]:
num_workers = 1
train_position_array = mp.Array('q', num_workers)
train_idx_array = mp.Array('q', num_workers)

resume_state = {}
if os.path.exists(os.path.join(checkpoint_path, save_file_name)):
    checkpoint = torch.load(os.path.join(checkpoint_path, save_file_name), map_location='cpu', weights_only=False)
    saved_offset = checkpoint.get('worker_byte_offsets', {})
    saved_idxs = checkpoint.get('worker_next_idx', {})
    resume_state = {
        w: (saved_offset.get(w, 0), saved_idxs.get(w, 0))
        for w in range(num_workers)
    }

In [7]:
train_data = ProteinDataset(
    corpus_path=train_corpus_path, tokenizer=tokenizer,
    context_length=config['context_length'], end_token=end_token,
    resume_state=resume_state, position_array=train_position_array,
    idx_array=train_idx_array
)
train_eval_dataset = ProteinDataset(
    corpus_path=train_corpus_path,
    tokenizer=tokenizer,
    context_length=config['context_length'],
    end_token="<|endofprotein|>",
    buffer_size=1,
)
val_data = ProteinDataset(corpus_path=val_corpus_path, tokenizer=tokenizer,
                         context_length=config['context_length'], end_token=end_token)

In [8]:
train_loader = DataLoader(
    train_data,
    batch_size=2,
    num_workers=num_workers,
    pin_memory=True,
)

train_eval_loader = DataLoader(
    train_eval_dataset,
    batch_size=2,
    num_workers=0,
)

val_loader = DataLoader(
    val_data,
    batch_size=2,
    num_workers=num_workers,
    pin_memory=True,
)

In [9]:
# model = GPT2(
#     emb_dim=config['emb_dim'],
#     d_out=config['emb_dim'],
#     vocab_size=config['vocab_size'],
#     context_length=config['context_length'],
#     num_heads=config['n_heads'],
#     n_layers=config['n_layers'],
#     dropout=config['drop_rate'],
#     qkv_bias=config['qkv_bias']
# )
model = LLAMA2(
    emb_dim=config['emb_dim'],
    vocab_size=config['vocab_size'],
    context_length=config['context_length'],
    num_heads=config['n_heads'],
    hidden_dim=config['hidden_dim'],
    n_layers=config['n_layers'],
    dtype=torch.bfloat16,
    qkv_bias=config['qkv_bias']
)


In [10]:
from torch.optim.lr_scheduler import LinearLR, CosineAnnealingLR, SequentialLR
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=0.1)

warmup_steps = 500
total_steps = 1_000

warmup_scheduler = LinearLR(
    optimizer,
    start_factor=0.01,
    end_factor=1.0,
    total_iters=warmup_steps
)

cosine_scheduler = CosineAnnealingLR(
    optimizer,
    T_max=total_steps - warmup_steps,
    eta_min=1e-5
)

scheduler = SequentialLR(
    optimizer,
    schedulers=[warmup_scheduler, cosine_scheduler],
    milestones=[warmup_steps]
)


In [11]:
train_losses, val_losses, track_tokens_seen = train_model(model, train_loader, val_loader, train_eval_loader,
             optimizer, scheduler, device,15, 250, 200, train_position_array, train_idx_array, num_workers, checkpoint_path, save_file_name)

Epoch 1/15: 0batch [00:08, ?batch/s, loss=10.500]

Epoch 1 (Step 0):
Train loss 10.545, Val loss 10.549
Train perplexity 37975.158, Val perplexity 38153.585


Epoch 1/15: 250batch [02:27,  3.04batch/s, loss=8.625] 

Epoch 1 (Step 250):
Train loss 8.486, Val loss 8.768
Train perplexity 4846.139, Val perplexity 6424.104


Epoch 1/15: 500batch [04:47,  3.03batch/s, loss=8.188]

...

Epoch 1 (Step 10750):
Train loss 8.530, Val loss 8.572
Train perplexity 5064.446, Val perplexity 5284.324


Epoch 1/15: 11000batch [1:42:18,  3.05batch/s, loss=6.438]

Epoch 1 (Step 11000):
Train loss 8.518, Val loss 8.554
Train perplexity 5003.098, Val perplexity 5186.166


Epoch 1/15: 11117batch [1:43:53,  1.78batch/s, loss=8.562]


KeyboardInterrupt: 

In [ ]:
torch.save(model.state_dict(), 'protbuildv1.pth')

In [ ]:
from ml.src.inference.sequence_generator import generate_protein

In [ ]:
state_dict = torch.load("protbuildv1.pth", weights_only=True)
model.load_state_dict(state_dict)
model = model.to(device)
model.eval()

GPT2(
  (tok_emb): Embedding(1200, 512)
  (pos_emb): Embedding(512, 512)
  (dropout): Dropout(p=0.1, inplace=False)
  (transformer_blocks): Sequential(
    (0): TransformerBlock(
      (norm1): Normalization()
      (attention): MultiHeadAttention(
        (Wquery): Linear(in_features=512, out_features=512, bias=True)
        (Wkey): Linear(in_features=512, out_features=512, bias=True)
        (Wvalue): Linear(in_features=512, out_features=512, bias=True)
        (proj): Linear(in_features=512, out_features=512, bias=True)
        (dropout): Dropout(p=0.1, inplace=False)
      )
      (dropout): Dropout(p=0.1, inplace=False)
      (norm2): Normalization()
      (ff): FeedForward(
        (layers): Sequential(
          (0): Linear(in_features=512, out_features=2048, bias=True)
          (1): GELU()
          (2): Linear(in_features=2048, out_features=512, bias=True)
        )
      )
    )
    (1): TransformerBlock(
      (norm1): Normalization()
      (attention): MultiHeadAttention(


In [ ]:
prompts = [
    "MKWVTFISLLLLFSSAYSRGVFRR",
    "MKTIIALSYIFCLVFAD",
    "MVLSPADKTNVKAAWGKVGA"
]

for prompt in prompts:
    print("=" * 80)
    print(prompt)

    output = generate_protein(
        model,
        prompt=prompt,
        tokenizer=tokenizer,
        max_new_tokens=100,
        context_size=config["context_length"],
        device=device,
        temperature=1.0,
        topk=25
    )

    print(output)

MKWVTFISLLLLFSSAYSRGVFRR
MKWVTFISLLLLFSSAYSRGVFRREPTDKDVYNSIHKFTQKRKNGKFNIYMTTQFMLNGDHRVGMLRKCYWCTHAKTQTHAMVYIHSSPCNEMSVTAQLLTLMDIPIYGVIAHMVFTVDEMKIPNDNYESC<|endofprotein|>
MKTIIALSYIFCLVFAD
MKTIIALSYIFCLVFADRAKMKDHKLWMNFSKSVECAQAHETIMKTHLDRSCKTASLFPLEGHCKICGMEQFMHAQHYDCMDFQQGFGCGGLQGKYTRGEIP<|endofprotein|>
MVLSPADKTNVKAAWGKVGA
MVLSPADKTNVKAAWGKVGAFTGIYRMPDENAIPGCKTRYNRCWGSEGFAAIGNRGYTTKWEAGL<|endofprotein|>


In [ ]:
temperatures = [0, 0.5, 0.8, 1.0, 1.2]

for temperature in temperatures:
    output = generate_protein(
        model,
        prompt="MVLSPADKTNVKAAWGKVGA",
        tokenizer=tokenizer,
        max_new_tokens=100,
        context_size=config["context_length"],
        device=device,
        temperature=temperature,
        topk=25
    )

    print(f"\nTemperature = {temperature}")
    print(output)


Temperature = 0
MVLSPADKTNVKAAWGKVGADHQHTIPHIQSQTKMTKEDHWDKCTQPYISCPYIMKHPIHRIMMTIMKFNFCYVYTVTTWIDFTQ<|endofprotein|>

Temperature = 0.5
MVLSPADKTNVKAAWGKVGADHLLCNEQNAQKDYIYKSYNHYYIKAVRRCGWPHFHKMMTKTIFMTYVANWWVL<|endofprotein|>

Temperature = 0.8
MVLSPADKTNVKAAWGKVGADHPTTAPIVAPFFWKANSMIAMPFCMHGNCGVISFNSAHETCGLMSVIDRIHVRWTWNDPLWGFQEDDHADWACFDWWGQETDSKWAHFFKKQTYRIQ<|endofprotein|>

Temperature = 1.0
MVLSPADKTNVKAAWGKVGAYNHEYNSAQGNDGNIDFMNINHFMFRVNCWLLSNTIHMTDFSEVLPETP<|endofprotein|>

Temperature = 1.2
MVLSPADKTNVKAAWGKVGANYTCKYYDYIWIIYMRRIMNAILSAKYMPHDQYQHQSECNTWAIGVYYFHDVFIPPRTEYVDVDNRRMCGSPFWGSQPEFNEHEWIDVFCA<|endofprotein|>
